In [0]:
import random
random.seed(11)

CATALOGO = "workspace"
departamentos = ["Valle", "Tolima", "Antioquia", "Quindio", "Narino"]

# Patrón estacional SINTÉTICO (mm/mes) — andino bimodal, estilizado
base_mm = {1:70, 2:80, 3:130, 4:200, 5:210, 6:110,
           7:70, 8:75, 9:120, 10:200, 11:190, 12:110}

# Desviación máx de la lluvia REAL vs. la norma, creciente por año
deriva = {2023: 0.10, 2024: 0.22, 2025: 0.38}

filas = []
for depto in departamentos:
    for anio in [2023, 2024, 2025]:
        for mes in range(1, 13):
            norma = base_mm[mes]
            pronosticada = round(norma * random.uniform(0.92, 1.08), 1)  # sigue la norma
            d = deriva[anio]
            real = round(norma * (1 + random.uniform(-d, d)), 1)          # se desvía, más cada año
            filas.append((depto, anio, mes, pronosticada, real))

cols = ["departamento", "anio", "mes", "precip_pronosticada_mm", "precip_real_mm"]
df_clima = spark.createDataFrame(filas, cols)

df_clima.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOGO}.bronze.clima_raw")
print("clima_raw creada:", df_clima.count(), "filas")   # 180
display(df_clima.orderBy("departamento", "anio", "mes").limit(12))

In [0]:
%sql
SELECT
  anio,
  ROUND(AVG(ABS(precip_real_mm - precip_pronosticada_mm)), 1)                                AS error_promedio_mm,
  ROUND(AVG(ABS(precip_real_mm - precip_pronosticada_mm) / precip_pronosticada_mm) * 100, 1) AS error_promedio_pct
FROM workspace.bronze.clima_raw
GROUP BY anio
ORDER BY anio;